# Notebook 10: Final audit, uncertainty and paper outputs

Use a **CPU runtime** and run the **single code cell**. It reads the completed Notebook 09 outputs automatically. No GPU, new dataset, training, model inference or new decision rule is required. Notebooks 08 and the raw action caches are not required for this audit.

It reproduces all 22 methods in all 15 configurations from the saved test decisions, checks all Notebook 09 aggregate metrics, and reports every baseline. It exports full CSV/LaTeX result tables, condition-level results, participant metrics, observed harm violations, descriptive F1–harm trade-offs, and PDF/PNG figures.

**Uncertainty:** The primary metric remains the mean over conditions, folds and seeds used in Notebook 09. Separately, participant-balanced Macro-F1 uses each participant's observed classes, averages that participant's saved conditions and three seeds, and bootstraps paired participants 10,000 times. All views of a participant remain together. Those intervals apply only to the secondary metric, conditional on the saved fits. They exclude retraining variability, do not resolve overlapping CV-training dependence, and are not simultaneous intervals or population guarantees. The notebook makes no p-value or significance claims. Seed SD is descriptive spread, not a confidence interval.

The five focal policies are compared with every non-oracle comparator. The notebook does not select a winning model or create a dataset-specific test-selected hybrid. The label-assisted single-candidate oracle is kept only as a diagnostic, excluded from trade-off frontiers and manuscript tables, and does not bound fusion or stacking.

Outputs: `SensorRepair_ICDSIS/outputs/notebook10_v1`.

Share the final printed output, `notebook10_comparison_summary.csv`, `notebook10_harm_audit.csv`, and `notebook10_run_report.json`. Then move to literature positioning and writing using the audited results. This notebook does not establish novelty or guarantee acceptance.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import os
import shutil
import zlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
SOURCE=PROJECT/'outputs'/'notebook09_v1'
OUTPUT=PROJECT/'outputs'/'notebook10_v1'
LOCAL=Path('/content/sensorrepair_notebook10_v1')
CODE_VERSION='sensorrepair_nb10_v1'
CELL_SIGNATURE='0ddd3c3c5126196feebe7d03e01ea353aa01b9f707d54fcfbd360001f62fe24b'
BOOTSTRAP_RESAMPLES=10000
BOOTSTRAP_SEED=20261021
METHODS=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct',
         'confidence_select','oracle_diagnostic','harm_aware_policy','benefit_only_policy',
         'confidence_margin_tuned','residual_mask_tuned','residual_reconstruct_tuned',
         'ternary_outcome_selector','utility_regression_selector',
         'uniform_fusion_safe','uniform_fusion_accuracy','global_fusion_safe','global_fusion_accuracy',
         'gated_fusion_safe','gated_fusion_accuracy','residual_stacking_safe','residual_stacking_accuracy']
FOCAL=['harm_aware_policy','gated_fusion_safe','gated_fusion_accuracy','residual_stacking_safe','residual_stacking_accuracy']
CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain','stuck'] for i in [1,2,3]]
SCOPES={'clean':['clean'],'known_faults':CONDITIONS[1:10],'unseen_stuck':CONDITIONS[10:]}
METRICS=['macro_f1','accuracy','decision_change_rate','helpful_rate','harmful_rate']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
print('NOTEBOOK 10 | final saved-prediction audit and paper outputs | CPU | one executable cell',flush=True)
print('No fitting, model inference, threshold changes, or new test decisions.',flush=True)
print('Primary results retain the original aggregation. Participant intervals are secondary and conditional.',flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'

def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'


def verified_copy(source,relative,expected):
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    temporary=target.with_name(target.name+'.partial')
    shutil.copyfile(source,temporary)
    assert file_sha256(temporary)==expected,f'Input hash mismatch: {source}'
    os.replace(temporary,target)
    return target


records=json.loads((SOURCE/'notebook09_artifact_inventory.json').read_text())
inventory={r['filename']:r for r in records}
assert len(records)==len(inventory)


def root_file(name):
    return verified_copy(SOURCE/name,'nb09/'+name,inventory[name]['sha256'])


report=json.loads(root_file('notebook09_run_report.json').read_text())
manifest=json.loads(root_file('notebook09_study_manifest.json').read_text())
assert report['study_complete'] and report['configurations_complete']==report['configurations_required']==15
assert report['study_signature']==manifest['signature']==json_signature(manifest['plan'])
JOBS=report['completed_results']
EXPECTED={(d,f'fold_{f}',s) for s in [2026,2027,2028] for d in ['dsads','har']
          for f in (range(4) if d=='dsads' else range(1))}
assert {(r['dataset'],r['fold'],r['seed']) for r in JOBS}==EXPECTED and len(JOBS)==15
original={name:pd.read_csv(root_file('notebook09_'+name+'.csv')) for name in
          ['per_condition','test_summary','aggregate_summary','per_seed_summary','comparisons','model_costs']}
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'source_study_signature':report['study_signature'],
      'source_report_sha256':inventory['notebook09_run_report.json']['sha256'],
      'primary_metric':report['aggregation'],'focal_policies':FOCAL,
      'comparators':'every non-oracle method other than the focal policy itself',
      'secondary_metric':'within-participant Macro-F1 on classes present; average scope conditions and all three saved seeds, then equal average participants',
      'bootstrap':'paired participant percentile; same participant indices for all methods/scopes; all seed/fault views remain with the participant',
      'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'bootstrap_seed':BOOTSTRAP_SEED,
      'bootstrap_limitations':'conditional on saved fits; excludes retraining variability; does not resolve overlapping CV training dependence; intervals not simultaneous; no population guarantee',
      'intervals_apply_to_primary_metric':False,'new_training_or_inference':False,'new_test_decisions':False,
      'descriptive_frontier':'maximize mean Macro-F1 and minimize observed harmful-change fraction; oracle excluded; no compute-cost or statistical dominance claim',
      'test_status':'exploratory post-analysis after test inspection; no model or threshold selection',
      'references':['https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html',
                    'https://jmlr.org/papers/v5/grandvalet04a.html']}
ANALYSIS_SIGNATURE=json_signature(plan)
path=OUTPUT/'notebook10_analysis_manifest.json'
if path.exists():assert json.loads(path.read_text())['signature']==ANALYSIS_SIGNATURE,'Analysis definition or source inputs changed.'
else:save_json(path,{'signature':ANALYSIS_SIGNATURE,'plan':plan})
print('Analysis plan recorded:',ANALYSIS_SIGNATURE[:16],flush=True)


def run_file(job,name):
    item=next(x for x in job['artifacts'] if x['filename']==name)
    relative='runs/'+identity(job)+'/'+name
    return verified_copy(SOURCE/relative,relative,item['sha256'])


def metrics(y,pred,base,K,present=None):
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    den=cm.sum(0)+cm.sum(1)
    f1=np.divide(2.*cm.diagonal(),den,out=np.zeros(K),where=den>0)
    correct=pred==y;was_correct=base==y
    helpful=float((~was_correct & correct).mean());harmful=float((was_correct & ~correct).mean())
    assert abs(correct.mean()-was_correct.mean()-helpful+harmful)<1e-12
    return {'macro_f1':float(f1.mean() if present is None else f1[present].mean()),'accuracy':float(correct.mean()),
            'decision_change_rate':float((pred!=base).mean()),'helpful_rate':helpful,'harmful_rate':harmful}


def reproduce():
    rows=[];participants=[];seen={};anchors={}
    for job in JOBS:
        info={k:job[k] for k in ['dataset','fold','seed']}
        assert job['original_prediction_parity'] and job['original_summary_parity']
        selection=json.loads(run_file(job,'selected_fusion.json').read_text())
        assert selection['signature']==job['signature']==json_signature({'study':report['study_signature'],**info})
        assert selection['selected_methods']==job['selected_methods']
        with np.load(run_file(job,'test_predictions.npz'),allow_pickle=False) as z:
            assert z['signature'].item()==job['signature']
            assert z['methods'].tolist()==METHODS and z['conditions'].tolist()==CONDITIONS
            pred=z['predictions'];y=z['y'];subjects=z['subjects'];ids=z['window_ids'];ix=z['source_indices']
            K=z['fusion_probabilities'].shape[-1]
        N=len(y)
        assert pred.shape==(len(METHODS),13,N) and N==job['test_windows']
        assert np.issubdtype(pred.dtype,np.integer) and pred.min()>=0 and pred.max()<K
        assert y.shape==subjects.shape==ids.shape==ix.shape==(N,)
        assert len(np.unique(ids))==N and y.min()>=0 and y.max()<K
        assert not set(subjects.tolist())&(set(selection['policy_participants'])|set(selection['validation_participants']))
        assert not set(ids.tolist())&set(selection['validation_window_ids'])
        for name,item in selection['selected_methods'].items():
            if item['settings']['alpha']==0:assert np.array_equal(pred[METHODS.index(name)],pred[0])
        key=(job['dataset'],job['seed']);people=set(subjects.tolist())
        assert not people & seen.setdefault(key,set()),'Test participant occurs in multiple folds.'
        seen[key]|=people
        anchor=(job['dataset'],job['fold'])
        if anchor in anchors:
            for a,b in zip(anchors[anchor],(y,subjects,ids,ix)):assert np.array_equal(a,b),'Cross-seed identity mismatch.'
        else:anchors[anchor]=tuple(x.copy() for x in (y,subjects,ids,ix))
        for m,method in enumerate(METHODS):
            for d,condition in enumerate(CONDITIONS):
                rows.append({**info,'method':method,'condition':condition,'windows':N,**metrics(y,pred[m,d],pred[0,d],K)})
            if method=='oracle_diagnostic':continue
            for subject in np.unique(subjects):
                use=subjects==subject;present=np.unique(y[use])
                for scope,names in SCOPES.items():
                    values=[metrics(y[use],pred[m,CONDITIONS.index(c),use],pred[0,CONDITIONS.index(c),use],K,present) for c in names]
                    average={k:float(np.mean([v[k] for v in values])) for k in METRICS}
                    average['macro_f1_present_classes']=average.pop('macro_f1')
                    participants.append({**info,'method':method,'subject':int(subject),'scope':scope,
                                         'windows':int(use.sum()),'classes_present':len(present),**average})
        print(f'  Reproduced {identity(job)}: {N} windows, {len(people)} participants.',flush=True)
    for d in ['dsads','har']:
        assert seen[d,2026]==seen[d,2027]==seen[d,2028]
        assert len(seen[d,2026])=={'dsads':8,'har':9}[d]
    return pd.DataFrame(rows),pd.DataFrame(participants)


def aggregate_metrics(rows):
    summaries=[]
    for scope,names in SCOPES.items():
        part=rows[rows.condition.isin(names)]
        keys=['dataset','fold','seed','method']
        assert part.groupby(keys).size().eq(len(names)).all()
        frame=part.groupby(keys,sort=False)[METRICS].mean().reset_index();frame['scope']=scope
        summaries.append(frame)
    summary=pd.concat(summaries,ignore_index=True)
    keys=['dataset','seed','method','scope']
    counts=summary.groupby(keys).size().reset_index(name='folds')
    assert counts.folds.eq(counts.dataset.map({'dsads':4,'har':1})).all()
    seed=summary.groupby(keys,sort=False)[METRICS].mean().reset_index()
    group=seed.groupby(['dataset','method','scope'],sort=False)
    means=group[METRICS].mean().add_suffix('_mean').reset_index()
    sd=group[METRICS].std(ddof=1).add_suffix('_seed_sd').reset_index()
    count=group.size().reset_index(name='complete_seeds');assert count.complete_seeds.eq(3).all()
    aggregate=means.merge(sd,on=['dataset','method','scope'],validate='one_to_one').merge(count,on=['dataset','method','scope'],validate='one_to_one')
    return summary,seed,aggregate


def bootstrap_delta(delta,indices):
    delta=np.asarray(delta,dtype=float)
    samples=delta[indices].mean(1)
    lo,hi=np.quantile(samples,[.025,.975])
    return float(delta.mean()),float(lo),float(hi),samples


def uncertainty(summary,seed,aggregate,participants):
    keys=['dataset','subject','method','scope']
    assert participants.groupby(keys).seed.nunique().eq(3).all()
    subject=participants.groupby(keys,sort=False).macro_f1_present_classes.mean().reset_index()
    lookup=aggregate.set_index(['dataset','method','scope'])
    output=[];deletion=[];run_rows=[]
    for dataset in ['dsads','har']:
        people=np.sort(subject[subject.dataset==dataset].subject.unique())
        rng=np.random.default_rng(BOOTSTRAP_SEED+zlib.crc32(dataset.encode()))
        indices=rng.integers(0,len(people),size=(BOOTSTRAP_RESAMPLES,len(people)),dtype=np.int16)
        saved=[];labels=[]
        for scope in SCOPES:
            part=subject[(subject.dataset==dataset)&(subject.scope==scope)]
            pivot=part.pivot(index='subject',columns='method',values='macro_f1_present_classes').loc[people]
            runs=summary[(summary.dataset==dataset)&(summary.scope==scope)].pivot(index=['fold','seed'],columns='method',values='macro_f1')
            seeds=seed[(seed.dataset==dataset)&(seed.scope==scope)].pivot(index='seed',columns='method',values='macro_f1').loc[[2026,2027,2028]]
            for method in FOCAL:
                for comparator in [m for m in METHODS if m not in [method,'oracle_diagnostic']]:
                    delta=100*(pivot[method]-pivot[comparator]).to_numpy()
                    point,lo,hi,samples=bootstrap_delta(delta,indices)
                    a,b=lookup.loc[(dataset,method,scope)],lookup.loc[(dataset,comparator,scope)]
                    run_delta=100*(runs[method]-runs[comparator]);seed_delta=100*(seeds[method]-seeds[comparator])
                    primary=100*(a.macro_f1_mean-b.macro_f1_mean)
                    assert abs(primary-seed_delta.mean())<1e-10
                    delete=(delta.sum()-delta)/(len(delta)-1)
                    for person,value in zip(people,delete):
                        deletion.append({'dataset':dataset,'scope':scope,'method':method,'comparator':comparator,
                            'removed_subject':int(person),'secondary_delta_pp':float(value),'models_refit':False})
                    for (fold,model_seed),value in run_delta.items():
                        run_rows.append({'dataset':dataset,'scope':scope,'method':method,'comparator':comparator,
                                         'fold':fold,'seed':int(model_seed),'delta_macro_f1_pp':float(value)})
                    output.append({'dataset':dataset,'scope':scope,'method':method,'comparator':comparator,
                        'primary_macro_f1_pct':100*a.macro_f1_mean,'comparator_macro_f1_pct':100*b.macro_f1_mean,
                        'primary_delta_macro_f1_pp':float(primary),'seed_delta_sd_pp':float(seed_delta.std(ddof=1)),
                        'positive_runs':int((run_delta>1e-10).sum()),'negative_runs':int((run_delta<-1e-10).sum()),
                        'tied_runs':int((run_delta.abs()<=1e-10).sum()),'total_runs':len(run_delta),
                        'participants':len(people),'secondary_participant_balanced_delta_pp':point,
                        'secondary_ci_low_pp':lo,'secondary_ci_high_pp':hi,
                        'secondary_deletion_min_pp':float(delete.min()),'secondary_deletion_max_pp':float(delete.max()),
                        'method_harm_pct':100*a.harmful_rate_mean,'comparator_harm_pct':100*b.harmful_rate_mean,
                        'interval_status':'secondary metric only; conditional, exploratory, not simultaneous; no significance or population guarantee'})
                    labels.append(scope+'|'+method+'|'+comparator);saved.append(samples)
        save_npz(OUTPUT/f'notebook10_{dataset}_paired_bootstrap.npz',subjects=people,participant_indices=indices,
                 comparison_labels=np.asarray(labels),delta_samples_pp=np.stack(saved),analysis_signature=np.asarray(ANALYSIS_SIGNATURE))
    return pd.DataFrame(output),subject,pd.DataFrame(deletion),pd.DataFrame(run_rows)


def pareto_rows(aggregate):
    rows=[]
    for keys,part in aggregate[aggregate.method.ne('oracle_diagnostic')].groupby(['dataset','scope'],sort=False):
        for _,r in part.iterrows():
            dominates=part[(part.macro_f1_mean>=r.macro_f1_mean-1e-12)&(part.harmful_rate_mean<=r.harmful_rate_mean+1e-12)&
                           ((part.macro_f1_mean>r.macro_f1_mean+1e-12)|(part.harmful_rate_mean<r.harmful_rate_mean-1e-12))]
            rows.append({'dataset':keys[0],'scope':keys[1],'method':r.method,'macro_f1_pct':100*r.macro_f1_mean,
                         'harm_pct':100*r.harmful_rate_mean,'descriptive_frontier':len(dominates)==0,
                         'dominated_by':';'.join(dominates.method.tolist()),'statistical_dominance_claim':False})
    return pd.DataFrame(rows)


LABELS={'keep':'Keep','confidence_select':'Confidence','harm_aware_policy':'Two heads',
        'ternary_outcome_selector':'Ternary','utility_regression_selector':'Utility selector',
        'gated_fusion_safe':'Gated fusion (conservative)','gated_fusion_accuracy':'Gated fusion (accuracy)',
        'residual_stacking_safe':'Stacking (conservative)','residual_stacking_accuracy':'Stacking (accuracy)'}


def make_figures(aggregate,comparisons):
    methods=['keep','confidence_select','harm_aware_policy','ternary_outcome_selector','gated_fusion_safe','residual_stacking_safe']
    colors=['#666666','#bf771d','#1c6799','#2d8450','#9b4291','#bb4242'];markers=['o','s','^','D','P','X']
    plt.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42})
    fig,axes=plt.subplots(2,2,figsize=(9.2,6.6))
    for row,dataset in enumerate(['dsads','har']):
        for col,scope in enumerate(['known_faults','unseen_stuck']):
            ax=axes[row,col]
            for method,color,marker in zip(methods,colors,markers):
                r=aggregate[(aggregate.dataset==dataset)&(aggregate.scope==scope)&(aggregate.method==method)].iloc[0]
                ax.scatter(100*r.harmful_rate_mean,100*r.macro_f1_mean,label=LABELS[method],color=color,marker=marker,s=45)
            ax.set(title=dataset.upper()+': '+('Known faults' if col==0 else 'Held-out stuck'),
                   xlabel='Harmful changes (% of all windows)',ylabel='Macro-F1 (%)')
            ax.grid(alpha=.15)
    fig.legend(*axes[0,0].get_legend_handles_labels(),loc='upper center',ncol=3,frameon=False,bbox_to_anchor=(.5,1.01))
    fig.tight_layout(rect=(0,.04,1,.90))
    fig.text(.5,.01,'Descriptive primary means; coincident methods may overlap. No uncertainty is shown.',ha='center',fontsize=8)
    for ext in ['png','pdf']:fig.savefig(OUTPUT/('notebook10_f1_harm.'+ext),dpi=200,bbox_inches='tight')
    plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(10,4.9))
    focal=['harm_aware_policy','gated_fusion_safe','residual_stacking_safe']
    for ax,dataset in zip(axes,['dsads','har']):
        y=0;tick=[];texts=[]
        for scope in ['known_faults','unseen_stuck']:
            for method in focal:
                r=comparisons[(comparisons.dataset==dataset)&(comparisons.scope==scope)&
                              (comparisons.method==method)&comparisons.comparator.eq('keep')].iloc[0]
                ax.plot([r.secondary_ci_low_pp,r.secondary_ci_high_pp],[y,y],color='#246b91',lw=1.6)
                ax.scatter(r.secondary_participant_balanced_delta_pp,y,color='#246b91',s=24)
                tick.append(y);texts.append(LABELS[method].replace(' (conservative)','')+' / '+('known' if scope=='known_faults' else 'stuck'));y+=1
        ax.axvline(0,color='#777777',ls='--',lw=.8);ax.set_yticks(tick,texts);ax.invert_yaxis()
        ax.set(title=dataset.upper(),xlabel='Participant-balanced F1 gain vs keep (pp)')
        ax.grid(axis='x',alpha=.15)
    fig.tight_layout(rect=(0,.10,1,1))
    fig.text(.5,.015,'Paired 95% participant percentile intervals, conditional on saved fits. Not primary-metric intervals or significance tests.',ha='center',fontsize=7.5)
    for ext in ['png','pdf']:fig.savefig(OUTPUT/('notebook10_secondary_intervals.'+ext),dpi=200,bbox_inches='tight')
    plt.close(fig)


def paper_tables(aggregate):
    paper=aggregate[aggregate.method.ne('oracle_diagnostic')].copy()
    for metric in METRICS:
        for suffix in ['_mean','_seed_sd']:paper[metric+suffix+'_pct']=100*paper[metric+suffix]
    columns=['dataset','method','scope','complete_seeds']+[m+s+'_pct' for m in METRICS for s in ['_mean','_seed_sd']]
    save_csv(OUTPUT/'notebook10_paper_table.csv',paper[columns])
    for dataset in ['dsads','har']:
        lines=[r'\begin{tabular}{lrrr}',r'\toprule',r'Method & Clean & Known faults & Stuck \\',r'\midrule']
        for method in [m for m in METHODS if m!='oracle_diagnostic']:
            cells=[]
            for scope in SCOPES:
                r=paper[(paper.dataset==dataset)&(paper.method==method)&(paper.scope==scope)].iloc[0]
                cells.append(f'${r.macro_f1_mean_pct:.2f} \\pm {r.macro_f1_seed_sd_pct:.2f}$')
            lines.append(LABELS.get(method,method).replace('_',r'\_')+' & '+' & '.join(cells)+r' \\')
        lines += [r'\bottomrule',r'\end{tabular}',
                  '% Macro-F1 percentages. Plus/minus is SD across three seed-level fold means, not a confidence interval.',
                  '% Requires booktabs. This includes all non-oracle methods; original test status and exploratory extensions must be described.']
        (OUTPUT/f'notebook10_{dataset}_table.tex').write_text('\n'.join(lines),encoding='utf-8')


def main():
    rows,participants=reproduce();summary,seed,aggregate=aggregate_metrics(rows)
    assert_table_parity(rows,original['per_condition'],['dataset','fold','seed','method','condition'],METRICS)
    assert_table_parity(summary,original['test_summary'],['dataset','fold','seed','method','scope'],METRICS)
    assert_table_parity(seed,original['per_seed_summary'],['dataset','seed','method','scope'],METRICS)
    assert_table_parity(aggregate,original['aggregate_summary'],['dataset','method','scope'],
                       [m+s for m in METRICS for s in ['_mean','_seed_sd']])
    comparisons,subject,deletion,runs=uncertainty(summary,seed,aggregate,participants)
    # Reconcile every Notebook 09 proposed-minus-comparator value as well.
    for _,r in original['comparisons'].iterrows():
        current=comparisons[(comparisons.dataset==r.dataset)&(comparisons.scope==r.scope)&
                            comparisons.method.eq('gated_fusion_'+r.profile)&comparisons.comparator.eq(r.comparator)].iloc[0]
        assert abs(current.primary_delta_macro_f1_pp-r.delta_macro_f1_pp)<1e-10
    harm=[]
    for keys,part in summary[summary.method.ne('oracle_diagnostic')].groupby(['dataset','method','scope'],sort=False):
        harm.append({**dict(zip(['dataset','method','scope'],keys)),'runs':len(part),
                     'runs_above_1pct_harm':int((part.harmful_rate>.01+1e-12).sum()),
                     'maximum_run_harm_pct':float(100*part.harmful_rate.max()),
                     'harm_guarantee':False})
    tradeoff=pareto_rows(aggregate)
    for name,frame in [('per_condition',rows),('test_summary',summary),('aggregate_summary',aggregate),
        ('per_seed_summary',seed),('participant_metrics',participants),('participant_balanced_scores',subject),
        ('comparison_summary',comparisons),('participant_deletion_sensitivity',deletion),('run_differences',runs),
        ('harm_audit',pd.DataFrame(harm)),('descriptive_frontier',tradeoff)]:save_csv(OUTPUT/('notebook10_'+name+'.csv'),frame)
    costs=original['model_costs'].groupby(['dataset','model'],sort=False).agg(
        configurations=('model','size'),parameters_min=('parameters','min'),parameters_max=('parameters','max'),
        median_training_seconds=('training_seconds','median'),total_training_seconds=('training_seconds','sum')).reset_index()
    save_csv(OUTPUT/'notebook10_meta_model_costs.csv',costs)
    paper_tables(aggregate);make_figures(aggregate,comparisons)
    notes=['# SensorRepair evidence for writing','',
        'All 22 methods and all 15 configurations were reproduced from the saved Notebook 09 predictions.',
        'Primary Macro-F1 retains the original averaging scheme. Report all stronger baselines.',
        'Original frozen evaluations and the later exploratory extensions must be distinguished.',
        'Do not choose different methods for DSADS and HAR using their test scores and call that combination a validation-selected algorithm.',
        '','# Gated fusion compared with keep and stronger controls','']
    focus=comparisons[comparisons.method.eq('gated_fusion_safe')&comparisons.scope.ne('clean')&
                      comparisons.comparator.isin(['keep','harm_aware_policy','ternary_outcome_selector','residual_stacking_safe'])]
    for _,r in focus.iterrows():
        notes.append(f'- {r.dataset}, {r.scope}, versus {r.comparator}: primary delta {r.primary_delta_macro_f1_pp:+.3f} pp; '
                     f'secondary participant delta {r.secondary_participant_balanced_delta_pp:+.3f} pp, '
                     f'conditional 95% interval [{r.secondary_ci_low_pp:+.3f}, {r.secondary_ci_high_pp:+.3f}] pp.')
    notes+=['','# Interpretation limits','',
        '- Participant intervals describe the secondary metric conditional on fixed fits, not primary-metric uncertainty.',
        '- Repeated seeds and synthetic fault views are not independent new participants.',
        '- Validation harm guards do not guarantee held-out harm below 1%. The accuracy profile has no such guard.',
        '- Small training losses do not establish held-out superiority.',
        '- The old label-assisted single-candidate oracle is not deployable and does not bound fusion or stacking.',
        '- An observed F1-harm frontier is descriptive and does not establish statistical or computational dominance.',
        '- The training cost table concerns the new meta-model fits. It does not measure end-to-end deployment latency.',
        '- Current evidence does not establish a novel method superior to all stronger baselines across both datasets.',
        '- Complete a primary-source literature comparison before making a novelty claim. Do not claim guaranteed acceptance.',
        '', '# Recommended scope','',
        'Freeze model development and write an empirical comparison of selective repair, preservation of correct predictions, and probability fusion under participant-disjoint evaluation.',
        'Retain negative findings and the stronger controls. A fresh method chosen from these results would require new confirmatory evidence.','']
    (OUTPUT/'notebook10_writing_notes.md').write_text('\n'.join(notes),encoding='utf-8')
    out={'code_version':CODE_VERSION,'analysis_signature':ANALYSIS_SIGNATURE,'source_study_signature':report['study_signature'],
        'configurations_reproduced':15,'methods_reproduced':22,'all_notebook09_metrics_and_comparisons_reproduced':True,
        'new_training':0,'new_model_inference':0,'new_test_decisions':0,'threshold_changes':0,
        'participants':{'dsads':8,'har':9},'primary_ci_claim':False,'secondary_intervals':plan['bootstrap_limitations'],
        'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'bootstrap_seed':BOOTSTRAP_SEED,'novelty_or_significance_claim':False,
        'model_selection_performed':False,'focal_comparisons':focus.to_dict('records'),'output_directory':str(OUTPUT)}
    save_json(OUTPUT/'notebook10_run_report.json',out)
    files=[{'filename':p.name,'bytes':p.stat().st_size,'sha256':file_sha256(p)} for p in sorted(OUTPUT.glob('notebook10_*'))
           if p.is_file() and p.name!='notebook10_artifact_inventory.json' and not p.name.endswith('.partial')]
    save_json(OUTPUT/'notebook10_artifact_inventory.json',files)
    print('\nNOTEBOOK 10 COMPLETE',flush=True)
    print('All 22 methods, 15 configurations, and Notebook 09 metrics/comparisons reproduced.',flush=True)
    print('GATED FUSION CONSERVATIVE | Primary deltas and separate secondary intervals (pp)',flush=True)
    print(focus[['dataset','scope','comparator','primary_delta_macro_f1_pp','secondary_participant_balanced_delta_pp',
                 'secondary_ci_low_pp','secondary_ci_high_pp','positive_runs','negative_runs','tied_runs']]
          .to_string(index=False,float_format=lambda x:f'{x:.3f}'),flush=True)
    print('Secondary intervals are conditional on saved fits and are not primary-score intervals or significance tests.',flush=True)
    print('New fitting/inference/decisions/threshold changes: 0 | GPU required: No',flush=True)
    print('Next: freeze experiments, position the evidence against related work, and write the paper with all stronger baselines.',flush=True)
    print('Outputs:',OUTPUT,flush=True)
    print('Share this final output and notebook10_comparison_summary.csv, notebook10_harm_audit.csv, notebook10_run_report.json.',flush=True)


main()


Mounted at /content/drive
NOTEBOOK 10 | final saved-prediction audit and paper outputs | CPU | one executable cell
No fitting, model inference, threshold changes, or new test decisions.
Primary results retain the original aggregation. Participant intervals are secondary and conditional.
Analysis plan recorded: 588cefe77f6d1537
  Reproduced dsads_fold_0_seed_2026: 2280 windows, 2 participants.
  Reproduced dsads_fold_1_seed_2026: 2280 windows, 2 participants.
  Reproduced dsads_fold_2_seed_2026: 2280 windows, 2 participants.
  Reproduced dsads_fold_3_seed_2026: 2280 windows, 2 participants.
  Reproduced har_fold_0_seed_2026: 2947 windows, 9 participants.
  Reproduced dsads_fold_0_seed_2027: 2280 windows, 2 participants.
  Reproduced dsads_fold_1_seed_2027: 2280 windows, 2 participants.
  Reproduced dsads_fold_2_seed_2027: 2280 windows, 2 participants.
  Reproduced dsads_fold_3_seed_2027: 2280 windows, 2 participants.
  Reproduced har_fold_0_seed_2027: 2947 windows, 9 participants.
  Rep